# Bonus task: Fine-tuning an ImageNet-pretrained MobileNetV2 on CIFAR-10

This is the **pretrained-model brownie point** for Project Trailcam. It is a separate runnable notebook, so the six original CNN experiments remain unchanged.

**Method:** Load torchvision's ImageNet-pretrained MobileNetV2; replace its 1000-class classifier with 10 outputs; fine-tune its last two feature blocks and the classifier. Compare the result against **MoreImprovedCNN with augmentation (83.87% test accuracy)** from the original project.

**Fairness:** We reuse the same CIFAR-10 images and deterministic 45,000/5,000 split (seed 4) used by MoreImprovedCNN. We resize to 96×96 and use ImageNet normalization because the backbone is pretrained; this is a transfer-learning comparison, not an equal-architecture or equal-compute comparison. The test set is evaluated **only after selecting the checkpoint on validation accuracy**.

**Run:** In Google Colab, select **Runtime → Change runtime type → T4 GPU** (if available), then run all cells. The notebook downloads the CIFAR-10 dataset and ImageNet weights. By default, Colab stores checkpoints and figures under `MyDrive/ACM_Trailcam/Transfer_Learning/` so interrupted sessions can resume. Change `EPOCHS` for longer training; don't report any accuracy until the evaluation cell actually runs.

In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2

SEED = 4
EPOCHS = 8
BATCH_SIZE = 64
TRAIN_PRETRAINED = True
SAVE_TO_DRIVE = "google.colab" in sys.modules

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = device.type == "cuda"
print(f"Device: {device}; PyTorch: {torch.__version__}; torchvision: {torchvision.__version__}")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/ACM_Trailcam/Transfer_Learning")
else:
    OUTPUT_DIR = Path("transfer_learning")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## Data: same split, pretrained-compatible preprocessing

Training examples: CIFAR-10's 45,000 samples drawn with `torch.Generator().manual_seed(4)`; validation: the remaining 5,000; test: the official 10,000-image test set. The augmented run uses the same **horizontal flip + padded random crop** family used in the original project. Resize to 96×96 for the ImageNet-pretrained model.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
eval_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Mirror used for the existing project if the default CIFAR-10 host is slow.
torchvision.datasets.CIFAR10.url = (
    "https://huggingface.co/datasets/MIT-OL-AI-D/"
    "cifar-10-python/resolve/main/cifar-10-python.tar.gz"
)

train_full = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=train_transform
)
val_full = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=False, transform=eval_transform
)
test_set = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=eval_transform
)

indices = torch.randperm(50000, generator=torch.Generator().manual_seed(SEED)).tolist()
train_ids, val_ids = indices[:45000], indices[45000:]
assert set(train_ids).isdisjoint(val_ids)

workers = 2 if device.type == "cuda" else 0
loader_kw = dict(num_workers=workers, pin_memory=(device.type == "cuda"))
train_loader_tl = DataLoader(Subset(train_full, train_ids), batch_size=BATCH_SIZE, shuffle=True, **loader_kw)
val_loader_tl = DataLoader(Subset(val_full, val_ids), batch_size=BATCH_SIZE, shuffle=False, **loader_kw)
test_loader_tl = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, **loader_kw)

classes = test_set.classes
print(f"Train {len(train_ids):,}, validation {len(val_ids):,}, test {len(test_set):,}")

## Transfer learning

We initialize from pretrained ImageNet weights, retain the early feature extraction filters, and **unfreeze the final two feature blocks** plus the replacement classifier. This is fine-tuning, not merely training a standalone linear classifier. Frozen early blocks keep their batch normalization running statistics fixed.

In [ ]:
weights = MobileNet_V2_Weights.DEFAULT
model = mobilenet_v2(weights=weights)
model.classifier[1] = nn.Linear(model.last_channel, 10)

for p in model.features.parameters():
    p.requires_grad = False
for block in model.features[-2:]:
    for p in block.parameters():
        p.requires_grad = True

model = model.to(device)
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_parameters = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_parameters:,} of {total_parameters:,} parameters")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([
    {"params": [p for b in model.features[-2:] for p in b.parameters()], "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 3e-4},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
best_path = OUTPUT_DIR / "mobilenetv2_best.pt"
last_path = OUTPUT_DIR / "mobilenetv2_last.pt"
history_path = OUTPUT_DIR / "mobilenetv2_history.json"

def evaluate_transfer(model, loader):
    model.eval()
    total_loss = total_correct = total = 0
    with torch.inference_mode():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            total_loss += criterion(logits, y).item() * y.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return total_loss / total, 100 * total_correct / total

In [ ]:
history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": []}
start_epoch, best_val_acc = 0, -1.0
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

if TRAIN_PRETRAINED and last_path.exists():
    saved = torch.load(last_path, map_location=device, weights_only=True)
    model.load_state_dict(saved["model_state"])
    optimizer.load_state_dict(saved["optimizer_state"])
    scheduler.load_state_dict(saved["scheduler_state"])
    if "scaler_state" in saved:
        scaler.load_state_dict(saved["scaler_state"])
    history = saved["history"]
    start_epoch = int(saved["epoch"])
    best_val_acc = float(saved["best_val_accuracy"])
    print(f"Resuming from epoch {start_epoch} / {EPOCHS}")

if TRAIN_PRETRAINED:
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        # Frozen earlier feature blocks must not update BatchNorm statistics.
        for block in model.features[:-2]:
            block.eval()

        running_loss = correct = seen = 0
        for x, y in train_loader_tl:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=(device.type == "cuda")):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            seen += y.size(0)

        train_loss = running_loss / seen
        train_accuracy = 100 * correct / seen
        val_loss, val_accuracy = evaluate_transfer(model, val_loader_tl)
        for k, value in [
            ("train_loss", train_loss), ("train_accuracy", train_accuracy),
            ("val_loss", val_loss), ("val_accuracy", val_accuracy)
        ]:
            history[k].append(value)

        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            torch.save(model.state_dict(), best_path)

        scheduler.step()
        torch.save({
            "epoch": epoch + 1, "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "history": history, "best_val_accuracy": best_val_acc
        }, last_path)
        history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")
        print(f"Epoch {epoch+1:2d}/{EPOCHS}: "
              f"train {train_accuracy:.2f}% | val {val_accuracy:.2f}% "
              f"| best val {best_val_acc:.2f}%", flush=True)

if not TRAIN_PRETRAINED and history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))

if not best_path.exists():
    raise FileNotFoundError(
        f"No trained checkpoint at {best_path}. Set TRAIN_PRETRAINED=True and run training first."
    )
model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
print(f"Loaded best validation checkpoint: {best_path}")

## Curves and held-out test result

The best checkpoint is selected using **validation accuracy only**. The test set is not used for checkpoint selection. Published `83.87%` is your existing custom-CNN benchmark from the six-run project; the new MobileNetV2 result will be generated by the next cell.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs_axis = np.arange(1, len(history["train_loss"]) + 1)
for ax, metric, label in [
    (axes[0], "loss", "Cross-entropy loss"),
    (axes[1], "accuracy", "Accuracy (%)")
]:
    ax.plot(epochs_axis, history[f"train_{metric}"], label="Train")
    ax.plot(epochs_axis, history[f"val_{metric}"], label="Validation")
    ax.set(xlabel="Epoch", ylabel=label, title=f"MobileNetV2: {label}")
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "mobilenetv2_training_curves.png", dpi=180)
plt.show()

test_loss, test_accuracy = evaluate_transfer(model, test_loader_tl)
BASELINE_TEST_ACC = 83.87
difference = test_accuracy - BASELINE_TEST_ACC
print(f"MoreImprovedCNN + augmentation: {BASELINE_TEST_ACC:.2f}% test accuracy")
print(f"Fine-tuned MobileNetV2:         {test_accuracy:.2f}% test accuracy")
print(f"Difference:                     {difference:+.2f} percentage points")

## Additional evidence: error analysis

This is also helpful for explaining when transfer learning works better and where it still fails. Images displayed here are **test** images; predictions are not used to tune hyperparameters.

In [ ]:
true_labels, predicted_labels, mistakes = [], [], []
model.eval()
with torch.inference_mode():
    for x, y in test_loader_tl:
        pred = model(x.to(device)).argmax(1).cpu()
        true_labels.extend(y.tolist())
        predicted_labels.extend(pred.tolist())
        for img, actual, guess in zip(x, y, pred):
            if actual.item() != guess.item() and len(mistakes) < 9:
                mistakes.append((img.cpu(), actual.item(), guess.item()))

matrix = confusion_matrix(true_labels, predicted_labels, labels=range(10))
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(matrix, display_labels=classes).plot(
    ax=ax, cmap="Blues", values_format="d", colorbar=False
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_title("MobileNetV2: CIFAR-10 test confusion matrix")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "mobilenetv2_confusion_matrix.png", dpi=180)
plt.show()

fig, axes = plt.subplots(3, 3, figsize=(10, 9))
for ax, item in zip(axes.flat, mistakes):
    img, actual, guess = item
    img = img.permute(1, 2, 0).numpy()
    img = np.clip(img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)
    ax.imshow(img)
    ax.set_title(f"Actual: {classes[actual]} | Predicted: {classes[guess]}")
    ax.axis("off")
for ax in axes.flat[len(mistakes):]:
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "mobilenetv2_misclassified.png", dpi=180)
plt.show()

In [ ]:
per_class_errors = matrix.sum(axis=1) - np.diag(matrix)
most_missed_index = int(np.argmax(per_class_errors))
result = {
    "model": "MobileNetV2",
    "pretraining": "ImageNet (torchvision default weights)",
    "fine_tuned_layers": "features[-2:] and classifier",
    "train_images": len(train_ids),
    "validation_images": len(val_ids),
    "test_images": len(test_set),
    "epochs_completed": len(history["train_accuracy"]),
    "best_validation_accuracy": round(max(history["val_accuracy"]), 2),
    "test_accuracy": round(test_accuracy, 2),
    "custom_cnn_test_accuracy": BASELINE_TEST_ACC,
    "difference_percentage_points": round(difference, 2),
    "most_missed_class": classes[most_missed_index],
    "most_missed_class_error_count": int(per_class_errors[most_missed_index])
}
(OUTPUT_DIR / "mobilenetv2_results.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
report = (
    "# Transfer-learning brownie point: MobileNetV2\n\n"
    "Fine-tuned torchvision's ImageNet-pretrained MobileNetV2 using the same "
    "CIFAR-10 45,000/5,000 training/validation split (seed 4) as MoreImprovedCNN. "
    "Inputs were resized to 96×96 and normalized with ImageNet statistics; "
    "the classifier and final two feature blocks were trainable. "
    f"The checkpoint with best validation accuracy ({result['best_validation_accuracy']:.2f}%) "
    f"achieved {test_accuracy:.2f}% on the official CIFAR-10 test set, "
    f"compared with {BASELINE_TEST_ACC:.2f}% for MoreImprovedCNN (augmented). "
    f"The difference was {difference:+.2f} percentage points.\n\n"
    "The pretrained model has an advantage from external ImageNet training and "
    "uses a different architecture/input resolution, so the result is a "
    "transfer-learning comparison rather than a controlled architecture-only ablation.\n"
)
(OUTPUT_DIR / "mobilenetv2_report.md").write_text(report, encoding="utf-8")
print(report)
print(f"Saved all outputs to: {OUTPUT_DIR}")

## Final upload to GitHub

After this notebook completes, transfer these files from `MyDrive/ACM_Trailcam/Transfer_Learning` into the GitHub repository's `transfer_learning/` folder: `mobilenetv2_best.pt`, `mobilenetv2_history.json`, `mobilenetv2_results.json`, `mobilenetv2_report.md`, `mobilenetv2_training_curves.png`, `mobilenetv2_confusion_matrix.png`, and `mobilenetv2_misclassified.png`.

The `mobilenetv2_last.pt` optimizer checkpoint is useful for recovering interrupted Colab sessions, but isn't required for the final submission. **Do not quote final test accuracy until this notebook has actually been run.**